In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('dataset.csv')
df.head()

,transaction_id,date,customer_id,amount,type,description
0,1,2020-10-26,926,6478.39,credit,Expect series shake art again our.
1,2,2020-01-08,466,1255.95,credit,Each left similar likely coach take.
2,3,2019-09-02,110,7969.68,debit,Direction wife job pull determine leader move ...
3,4,2020-12-02,142,2927.41,credit,Agree reveal buy black already.
4,5,2020-12-02,944,4661.88,debit,Child relationship show college whom speech.


In [2]:
# Convert date
df['date'] = pd.to_datetime(df['date'])

# Extract useful features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# Encode type
df['type'] = df['type'].map({'credit': 1, 'debit': 0})

# Select features
features = ['amount', 'type', 'year', 'month', 'day']
X = df[features]

In [3]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X)

# Predictions
df['anomaly'] = model.predict(X)  # -1 = anomaly, 1 = normal

In [5]:
import joblib

joblib.dump(model, 'isolation_model.pkl')

['isolation_model.pkl']

In [6]:
# Convert predictions
df['anomaly_flag'] = df['anomaly'].map({1: 0, -1: 1})

# Fake ground truth (example: very high amounts = anomaly)
df['true_label'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df['true_label'], df['anomaly_flag'])
print("Model Accuracy:", accuracy)

Model Accuracy: 0.91378


In [11]:
import pandas as pd
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('dataset.csv')

# --- Preprocessing ---
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.weekday

df['customer_freq'] = df['customer_id'].map(df['customer_id'].value_counts())
df['type'] = df['type'].map({'credit': 1, 'debit': 0})
df['desc_length'] = df['description'].apply(len)

# --- Features (NOW MORE POWERFUL) ---
features = [
    'amount', 'type', 'year', 'month', 'day',
    'weekday', 'customer_freq', 'desc_length'
]

X = df[features]

# --- Pipeline ---
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', IsolationForest(contamination=0.05, random_state=42))
])

pipeline.fit(X)

# Save full pipeline
joblib.dump(pipeline, 'isolation_pipeline.pkl')

['isolation_pipeline.pkl']